# Phase 3 - Machine Learning Models

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load Data from Phase 2

In [3]:
# Load Phase 2 output (all 3 outcomes included)
df = pd.read_csv("../data/processed/modeling_dataset_phase3.csv")
df["date"] = pd.to_datetime(df["date"])

# Encode target
outcome_map = {"home_win": 1, "draw": 0, "away_win": -1}
df["outcome"] = df["result"].map(outcome_map)

print("Unmapped rows:", df["outcome"].isna().sum())
print("\nOutcome distribution:")
print(df["outcome"].value_counts().sort_index())
print("\nAs percentages:")
print(df["outcome"].value_counts(normalize=True).mul(100).round(2).sort_index())

display(df.head())

Unmapped rows: 0

Outcome distribution:
outcome
-1     6654
 0     5683
 1    11428
Name: count, dtype: int64

As percentages:
outcome
-1    28.00
 0    23.91
 1    48.09
Name: proportion, dtype: float64


,date,home_team,away_team,neutral,rank_diff,points_diff,form_diff_5,result,outcome
0,1993-01-28,Cameroon,Finland,True,22.0,12.0,-0.666667,home_win,1
1,1993-01-30,India,Cameroon,False,-121.0,-41.0,-1.166667,draw,0
2,1993-02-26,Algeria,Ghana,False,9.0,5.0,-0.266667,home_win,1
3,1993-03-07,Honduras,Costa Rica,False,-3.0,-3.0,-3.000000,home_win,1
4,1993-03-09,Honduras,El Salvador,False,10.0,4.0,1.166667,home_win,1


## Feature Engineer

Features being built:
- `rank_diff` and `points_diff` → already in dataset from Phase 2
- `neutral` → already in dataset from Phase 2
- `avg_goals_scored_5` → rolling avg goals scored (last 5 matches)
- `avg_goals_conceded_5` → rolling avg goals conceded (last 5 matches)
- `recent_win_rate_5` → win rate in last 5 matches

All rolling features use shift(1) to prevent data leakage.

In [5]:
# Step 1: Load raw match history and build a team-level table 
raw = pd.read_csv("../data/processed/results_cleaned.csv")
raw["date"] = pd.to_datetime(raw["date"])

# Each match becomes two rows: one for home team, one for away team
home_side = raw[["date", "home_team", "home_score", "away_score"]].copy()
home_side.columns = ["date", "team", "goals_for", "goals_against"]

away_side = raw[["date", "away_team", "away_score", "home_score"]].copy()
away_side.columns = ["date", "team", "goals_for", "goals_against"]

# Combine home and away sides
team_hist = pd.concat([home_side, away_side], ignore_index=True)
team_hist = team_hist.sort_values(["team", "date"]).reset_index(drop=True)

# Win = 1 if the team scored more foals than the opponent
team_hist["win"] = (team_hist["goals_for"] > team_hist["goals_against"]).astype(int)

print("Team History shape: ", team_hist.shape)
display(team_hist.head(8))

Team History shape:  (48368, 5)


,date,team,goals_for,goals_against,win
0,2003-03-18,Afghanistan,0,4,0
1,2003-11-19,Afghanistan,0,11,0
2,2003-11-23,Afghanistan,0,2,0
3,2005-11-09,Afghanistan,0,4,0
4,2005-12-07,Afghanistan,1,9,0
5,2005-12-09,Afghanistan,0,1,0
6,2005-12-11,Afghanistan,2,1,1
7,2006-04-05,Afghanistan,1,1,0


In [6]:
# Step 2: Build rolling features (last 5 matches, no leakage)
def rolling_mean(series):
    return series.shift(1).rolling(5, min_periods=3).mean()

team_hist["avg_goals_scored_5"] = team_hist.groupby("team")["goals_for"].transform(rolling_mean)
team_hist["avg_goals_conceded_5"] = team_hist.groupby("team")["goals_against"].transform(rolling_mean)
team_hist["recent_win_rate_5"] = team_hist.groupby("team")["win"].transform(rolling_mean)

display(team_hist[["date", "team", "avg_goals_scored_5", "avg_goals_conceded_5", "recent_win_rate_5"]].head(12))

,date,team,avg_goals_scored_5,avg_goals_conceded_5,recent_win_rate_5
0,2003-03-18,Afghanistan,NaN,NaN,NaN
1,2003-11-19,Afghanistan,NaN,NaN,NaN
2,2003-11-23,Afghanistan,NaN,NaN,NaN
3,2005-11-09,Afghanistan,0.0,5.666667,0.0
4,2005-12-07,Afghanistan,0.0,5.250000,0.0
5,2005-12-09,Afghanistan,0.2,6.000000,0.0
6,2005-12-11,Afghanistan,0.2,5.400000,0.0
7,2006-04-05,Afghanistan,0.6,3.400000,0.2
8,2007-10-08,Afghanistan,0.8,3.200000,0.2
9,2007-10-26,Afghanistan,0.8,3.000000,0.2


In [7]:
# Step 3: Merge features back into the ,atch level dataset
feats = team_hist[["date", "team", "avg_goals_scored_5", "avg_goals_conceded_5", "recent_win_rate_5"]]

# Attach home team features
df = df.merge( feats.rename(columns={
    "team": "home_team",
    "avg_goals_scored_5": "home_avg_goals_scored_5",
    "avg_goals_conceded_5": "home_avg_goals_conceded_5",
    "recent_win_rate_5": "home_recent_win_rate_5"
}), on=["date", "home_team"], how="left")
# Attach away team features
df = df.merge( feats.rename(columns={
    "team": "away_team",
    "avg_goals_scored_5": "away_avg_goals_scored_5",
    "avg_goals_conceded_5": "away_avg_goals_conceded_5",
    "recent_win_rate_5": "away_recent_win_rate_5"
}), on=["date", "away_team"], how="left")

# Difference features: postive = home team is stronger
df["avg_gf_diff"] = df["home_avg_goals_scored_5"] - df["away_avg_goals_scored_5"]
df["avg_ga_diff"] = df["home_avg_goals_conceded_5"] - df["away_avg_goals_conceded_5"]
df["win_rate_diff"] = df["home_recent_win_rate_5"] - df["away_recent_win_rate_5"]
df["is_neutral"] = df["neutral"].astype(int)

print("Shape after merge:", df.shape)
print("\nMissing values in new features:")
print(df[["avg_gf_diff", "avg_ga_diff", "win_rate_diff"]].isna().sum())

display(df[["date", "home_team", "away_team",
            "rank_diff", "avg_gf_diff", "avg_ga_diff", "win_rate_diff", "is_neutral", "outcome"]].head(10))

Shape after merge: (23793, 19)

Missing values in new features:
avg_gf_diff      0
avg_ga_diff      0
win_rate_diff    0
dtype: int64


,date,home_team,away_team,rank_diff,avg_gf_diff,avg_ga_diff,win_rate_diff,is_neutral,outcome
0,1993-01-28,Cameroon,Finland,22.0,-0.333333,0.333333,-0.333333,1,1
1,1993-01-30,India,Cameroon,-121.0,-0.416667,0.750000,-0.250000,0,0
2,1993-02-26,Algeria,Ghana,9.0,-0.200000,0.066667,-0.133333,0,1
3,1993-03-07,Honduras,Costa Rica,-3.0,-1.666667,1.333333,-0.666667,0,1
4,1993-03-09,Honduras,El Salvador,10.0,0.833333,-0.333333,0.500000,0,1
5,1993-03-12,El Salvador,Bolivia,37.0,-0.833333,0.416667,-0.333333,0,0
6,1993-03-14,Honduras,Bolivia,47.0,0.300000,-0.700000,0.350000,0,0
7,1993-03-16,Honduras,Bolivia,47.0,0.400000,-0.600000,0.400000,0,0
8,1993-03-31,Poland,Lithuania,73.0,0.000000,-2.333333,-0.333333,0,0
9,1993-04-04,El Salvador,Mexico,-25.0,0.466667,0.133333,0.000000,0,1


In [9]:
# Step 4: Keep only rows where ALL features are present
feature_cols = ["rank_diff", "points_diff", "form_diff_5", "avg_gf_diff", "avg_ga_diff", "win_rate_diff", "is_neutral"]

df_model = df.dropna(subset=feature_cols + ["outcome"]).copy().reset_index(drop=True)

print("Rows available for modeling:", len(df_model))
print("\nOutcome distribution in modeling dataset:")
print(df_model["outcome"].value_counts().sort_index())
print("\nAs percentages:")
print(df_model["outcome"].value_counts(normalize=True).mul(100).round(2).sort_index())

Rows available for modeling: 23793

Outcome distribution in modeling dataset:
outcome
-1     6658
 0     5685
 1    11450
Name: count, dtype: int64

As percentages:
outcome
-1    27.98
 0    23.89
 1    48.12
Name: proportion, dtype: float64


## Train/Test Split by Year
Why split by year, not randomly?

If you split randomly, a match from 2010 could end up in your training set using form features built from 2012 matches. That is data leakage — the model would be learning from the future.

Splitting by year guarantees:

training data = everything before the split year
test data = everything from the split year onwards

This simulates a real scenario: "I trained on historical data, now I predict future matches."